# Normalized Product (NormCovar) from local files

This notebook runs the full NormProd processing pipeline using local GeoTIFFs.
Requires the user to create a directory called "DATA" in the "examples" folder. 

# 1. Imports

In [ ]:
import pathlib
import sys
import urllib.request
from pathlib import Path

import numpy as np
from osgeo import gdal
from normalized_product import normcovar, normcovar_utils
import re


# 2. Parameters

Set up file pathing

In [ ]:
DATA_DIR = pathlib.Path("./data")
site = "test"

GEOTIFF_DIR = DATA_DIR / site
GEOTIFF_DIR.mkdir(parents=True, exist_ok=True)

test_scene_1_s3_path = "https://data.dev.dea.ga.gov.au/experimental/baseline/mcmurdo/ga_s1_nrb_ew_hh_hv_1/2019/12/17/S1A_EW_GRDM_1SDH_20191217T112553_20191217T112658_030387_037A1C_58E9/ga_s1a_nrb_1-0-0__EW___A_20191217T112553_HH-gamma0_db.tif"
test_scene_2_s3_path = "https://data.dev.dea.ga.gov.au/experimental/baseline/mcmurdo/ga_s1_nrb_ew_hh_hv_1/2019/12/29/S1A_EW_GRDM_1SDH_20191229T112553_20191229T112657_030562_038020_E608/ga_s1a_nrb_1-0-0__EW___A_20191229T112553_HH-gamma0_db.tif"

for url in [test_scene_1_s3_path,test_scene_2_s3_path]:
    filename = url.split("/")[-1]
    dest = GEOTIFF_DIR / filename
    if not dest.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, dest)
        print(f"Saved to {dest}")

# 3. Load and prepare data

`normprod_utils.check_and_trim_image_pair`

Manually set date and supply to check and trim function

In [ ]:
# Test pair: S1_image_pair_20210708T145617_20210720T145618
image_1 = GEOTIFF_DIR / Path(test_scene_1_s3_path).name
image_2 = GEOTIFF_DIR / Path(test_scene_2_s3_path).name 
img_pair = [image_1, image_2]

date1 = re.search(r"(\d{8}T\d{6})", str(Path(test_scene_1_s3_path).name)).group(1)
date2 = re.search(r"(\d{8}T\d{6})", str(Path(test_scene_2_s3_path).name)).group(1)

# Define output directory
IMG_PAIR_DIR = GEOTIFF_DIR / f"S1_image_pair_{date1}_{date2}"
print(image_1)
print(image_2)
print(IMG_PAIR_DIR)

In [ ]:
# Temporal baseline in days
min_temp_baseline = 11.9
max_temp_baseline = 12.1

# Output epsg
output_epsg = 3031

In [ ]:
normcovar_utils.check_and_trim_image_pair(
    img_pair,
    IMG_PAIR_DIR,
    min_temp_baseline = min_temp_baseline,
    max_temp_baseline = max_temp_baseline,
    output_epsg = output_epsg,
    date1 = date1,
    date2 = date2,
    overwrite = False,
)


# 4. Process image pair

In [ ]:
# Define window sizes to process
window_list = [11,21,33]

# Save intermediate normprod steps?
save_intermediate_products = True

# Define min/max values for normprod scaling to RGB image
NP_min = -0.5
NP_max = 1.0

# Resample NP RGB image
resample = True

# Set resamping interval for NP RGB image and landmask
resample_interval = 10

In [ ]:
normcovar.fully_process_single_image_pair(
    IMG_PAIR_DIR,
    windows = window_list,
    save_intermediate_products = save_intermediate_products,
    NP_min = NP_min,
    NP_max = NP_max,
    landmask_shapefile_path = None,
    erode_landmask = None,
    resample = resample,
    resample_interval = resample_interval,
)
